# Mission d'adaptation : un nouveau site, sans code fourni
**ÉNONCÉ**

Vous savez maintenant scraper et crawler un site (Books to Scrape), et le faire poliment. **Cette mission change de site, et on ne vous donne aucun code de collecte.** C'est à vous d'adapter votre démarche à une structure différente.

**Le site :** `quotes.toscrape.com`, des citations célèbres, créé lui aussi pour s'entraîner.

## Ce que vous devez produire
1. **`citations`** : un tableau de **toutes** les citations (toutes les pages), avec les colonnes `texte`, `auteur`, `tags`, `url_page`, `url_auteur`.
2. **`auteurs`** : un tableau de la fiche de **chaque auteur, une seule fois**, avec `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`.
3. **`nb_requetes`** : le nombre de pages que votre script a demandées (pages de citations + fiches d'auteurs, sans compter `robots.txt`).
4. **`citations_completes`** : les citations enrichies de la fiche de leur auteur.
5. **`top_auteurs`** et **`top_tags`** : les 5 auteurs les plus cités et les 10 tags les plus fréquents.
6. **Une note de 5 lignes** : ce qui change par rapport à Books to Scrape, et comment vous l'avez géré.

## Règles
Les règles du notebook précédent s'appliquent : vous vous **présentez** (User-Agent avec contact), vous **lisez le robots.txt**, vous faites une **pause**. Ces outils vous sont fournis plus bas.

## Pour vous mettre sur la piste (sans code)
- **Inspectez** le site dans votre navigateur avant d'écrire quoi que ce soit. Qu'est-ce qui est pareil ? Qu'est-ce qui est différent ?
- Regardez **la forme des adresses** des liens « Next » et « (about) ».
- Un **même auteur** apparaît dans de nombreuses citations. Faut-il télécharger sa fiche à chaque fois ?
- Une citation a **plusieurs tags**. Comment stocker cela dans une seule case ?
- Le texte d'une citation est entouré de **guillemets typographiques**. Faut-il les garder ?

In [ ]:
import re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"      # un NOUVEAU site-école, à la structure différente de Books to Scrape

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Se présenter
Comme dans le notebook précédent : remplacez `___` par votre contact (adresse e-mail ou adresse d'une page de projet).

In [ ]:
ENTETES = {"User-Agent": "TPFormation/1.0 (___) python-requests"}   # TODO : remplacez ___ par votre contact
print(ENTETES)

In [ ]:
# Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact')

## Étape 2 — Vos outils (fournis)
La fonction de requête polie et la lecture du `robots.txt`, issues du notebook précédent.

In [ ]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)
        reponse = requests.get(url, headers=ENTETES, timeout=20)
        if reponse.status_code in (429, 503):
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser(); regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 3 — Consulter le robots.txt
Lisez le `robots.txt` de ce site, puis demandez s'il est permis de visiter la page 2 des citations (`/page/2/`) et la fiche d'un auteur (`/author/Albert-Einstein`). Stockez les deux réponses (vrai ou faux) dans `autorise_pages` et `autorise_auteur`.

In [ ]:
regles = lire_robots(SITE)
# TODO : calculer autorise_pages et autorise_auteur (si regles vaut None, aucune restriction annoncée)


In [ ]:
# Contrôle (ne pas modifier)
controle(isinstance(autorise_pages, bool) and isinstance(autorise_auteur, bool), 'Autorisations établies : pages=%s, auteur=%s' % (autorise_pages, autorise_auteur), 'Attendu : deux booléens')

## Étape 4 — Collecter toutes les citations *(à écrire seul)*
Objectif : la variable **`citations`** (un `DataFrame`) et le compteur **`nb_requetes`**. Cette cellule est **vide** : à vous d'écrire la boucle.

In [ ]:
# À vous : collecter toutes les citations (variables attendues : citations, nb_requetes)
nb_requetes = 0


In [ ]:
# Contrôle (ne pas modifier)
colonnes = {"texte", "auteur", "tags", "url_page", "url_auteur"}
controle(isinstance(citations, pd.DataFrame) and colonnes <= set(citations.columns) and len(citations) >= 10,
         "%d citations collectées, colonnes correctes" % len(citations), "Attendu : un DataFrame avec les colonnes " + ", ".join(sorted(colonnes)) + " et au moins 10 lignes")
controle(citations["tags"].map(lambda t: isinstance(t, list)).all() and not citations["texte"].str.startswith(("“", '"')).any(),
         "Les tags sont des listes et les guillemets typographiques sont retirés", "Les tags doivent être des listes, et le texte ne doit plus commencer par un guillemet")
controle(citations["url_auteur"].str.startswith("http").all() and citations["url_page"].str.startswith("http").all(),
         "Toutes les adresses sont complètes", "Les adresses doivent commencer par http : pensez aux liens relatifs")
controle(not citations.duplicated(subset=["texte", "auteur"]).any(), "Aucune citation en double", "Il y a des citations en double : une page a-t-elle été lue deux fois ?")

## Étape 5 — Collecter la fiche de chaque auteur *(à écrire seul)*
Objectif : la variable **`auteurs`**, avec **une seule ligne par auteur**. Inspectez une fiche d'auteur pour trouver où sont la date et le lieu de naissance, et la biographie. **Attention au nombre de requêtes** : le contrôle suivant le vérifie.

In [ ]:
# À vous : collecter une fiche par auteur (variable attendue : auteurs)


In [ ]:
# Contrôle (ne pas modifier)
colonnes_a = {"url_auteur", "auteur", "date_naissance", "lieu_naissance", "biographie"}
controle(isinstance(auteurs, pd.DataFrame) and colonnes_a <= set(auteurs.columns) and auteurs["url_auteur"].is_unique,
         "%d fiches d'auteurs, une seule par auteur" % len(auteurs), "Attendu : les colonnes " + ", ".join(sorted(colonnes_a)) + " et une seule ligne par auteur")
controle(set(citations["url_auteur"]) <= set(auteurs["url_auteur"]), "Tous les auteurs cités ont une fiche", "Certains auteurs cités n'ont pas de fiche")
attendu = citations["url_page"].nunique() + citations["url_auteur"].nunique()
controle(nb_requetes <= attendu + 2, "%d requêtes pour %d attendues au minimum : fiches non redemandées" % (nb_requetes, attendu), "%d requêtes alors que %d suffisent : un auteur a été redemandé plusieurs fois" % (nb_requetes, attendu))

## Étape 6 — Fusionner et analyser *(à écrire seul)*
Objectifs : **`citations_completes`** (chaque citation avec la fiche de son auteur), **`top_auteurs`** (les 5 auteurs les plus cités) et **`top_tags`** (les 10 tags les plus fréquents). Indice : pour compter des tags stockés dans des listes, il faut d'abord « déplier » ces listes.

In [ ]:
# À vous : citations_completes, top_auteurs, top_tags


In [ ]:
# Contrôle (ne pas modifier)
controle(len(citations_completes) == len(citations) and "date_naissance" in citations_completes.columns and citations_completes["date_naissance"].notna().all(),
         "Fusion correcte : chaque citation a la fiche de son auteur", "La fusion doit garder toutes les citations, chacune avec la date de naissance de son auteur")
controle(len(top_auteurs) == min(5, citations["auteur"].nunique()) and len(top_tags) == min(10, citations.explode("tags")["tags"].nunique()) and top_auteurs.is_monotonic_decreasing and top_tags.is_monotonic_decreasing,
         "Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent", "Attendu : 5 auteurs et 10 tags triés par fréquence décroissante")

## Étape 7 — Votre note (5 lignes)
Répondez ici, en 5 lignes maximum :

1. Ce qui était **identique** à Books to Scrape.
2. Ce qui était **différent**.
3. Ce que vous avez dû **changer** dans votre démarche.
4. Comment vous avez évité de **redemander** les fiches d'auteurs.
5. Ce qui, à votre avis, **casserait** votre script si le site changeait.

> 